# SDXL Luxury LoRA: entrenamiento y experimentos reproducibles

Se entrenó un adaptador propio sobre SDXL con fotografías reales; los pesos de la base permanecen congelados.
Este notebook recupera la evidencia del entrenamiento y ejecuta una comparación controlada de inferencia.
El entrenamiento costoso se reutiliza por defecto. Para repetirlo en Kaggle, cambia `USE_CACHED_TRAINING`.
Los resultados visuales deben revisarse antes de elegir una configuración final.

**Datos:** 133 fotos, 113 train / 10 validación / 10 prueba; 3 dormitorios ordinarios adicionales para desarrollo img2img.
**Evidencia fotográfica:** procedencia, licencia por archivo y metadatos de cámara; predominan hoteles.
**Anotaciones:** borradores Qwen3.5 revisados por el asistente; las referencias de estilo las eligió el usuario.

## 1. Entorno y entradas

En Kaggle, adjuntar el dataset privado de entrenamiento y los outputs de los kernels de entrenamiento y baseline.
El archivo de metadatos proporcionado configura esas dependencias. No se requieren secretos para descargar los modelos públicos.
Las versiones medidas fueron PyTorch 2.11.0, Diffusers 0.42.0, Transformers 5.20.2 y PEFT 0.21.2.
El código elimina `torchao` incompatible únicamente en el entorno temporal de Kaggle; LoRA estándar no usa ese paquete.

In [ ]:
from pathlib import Path
import csv, json, hashlib, importlib.util, runpy, shutil
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage

ROOT = Path.cwd()
IN_KAGGLE = Path('/kaggle/input').exists()
SEARCH = Path('/kaggle/input') if IN_KAGGLE else ROOT
USE_CACHED_TRAINING = True
print('Modo:', 'Kaggle' if IN_KAGGLE else 'análisis local de resultados guardados')

## 2. Validación del conjunto

Cada grupo permanece en una sola partición. Los grupos se obtuvieron de forma conservadora mediante títulos,
autores y revisión de similitud; no se verificaron independientemente todas las identidades físicas de las habitaciones.

In [ ]:
manifest = next(SEARCH.rglob('manifest.csv')) if IN_KAGGLE else ROOT / 'dataset' / 'manifest.csv'
with manifest.open(encoding='utf-8-sig') as f:
    rows = list(csv.DictReader(f))
counts = {split: sum(r['split'] == split for r in rows) for split in ['train', 'validation', 'test']}
assert counts['train'] >= 100
groups = {}
for row in rows:
    groups.setdefault(row['room_group'], set()).add(row['split'])
assert all(len(splits) == 1 for splits in groups.values())
print('Particiones:', counts)
print('Autores:', len({r['author'] for r in rows}))
print('Grupos sin cruce:', len(groups))

## 3. Entrenamiento propio

LoRA de rango 8 en las proyecciones de atención del UNet, tasa inicial máxima 1e-4, FP16,
acumulación de 2 y 300 iteraciones. VAE y codificadores de texto congelados.
Las condiciones se precalculan; se mantiene la proporción de las fotos con dimensiones múltiplos de ocho.
Se recorta un margen inferior pequeño en fotos con marcas de fecha conocidas; el reporte enumera los casos.
La receta adapta el [entrenador oficial de Diffusers](https://github.com/huggingface/diffusers/blob/v0.42.0/examples/text_to_image/train_text_to_image_lora_sdxl.py).
Los archivos guardados contienen el adaptador aprendido, no una copia de SDXL.

In [ ]:
reports = list(SEARCH.rglob('training_report.json'))
complete = [p for p in reports if json.loads(p.read_text())['status'] == 'complete']
if USE_CACHED_TRAINING and complete:
    training_report = json.loads(complete[0].read_text())
    print('Evidencia de entrenamiento recuperada:', complete[0])
elif IN_KAGGLE:
    trainer = next(SEARCH.rglob('train_cached_sdxl.py'))
    spec = importlib.util.spec_from_file_location('luxury_training', trainer)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.train()
    training_report = json.loads((ROOT / 'training_report.json').read_text())
else:
    raise RuntimeError('Falta el entrenamiento completo; ejecutar el kernel de Kaggle.')
assert training_report['status'] == 'complete'
assert training_report['train_count'] >= 100
assert hashlib.sha256(manifest.read_bytes()).hexdigest() == training_report['manifest_sha256']
print('Pasos:', training_report['step'], '| actualizaciones:', training_report['optimizer_updates'])
print('Parámetros entrenables:', training_report['trainable_parameters'])
print('Configuración:', training_report['config'])

## 4. Curva de pérdida

La pérdida depende también del ruido, timestep y fotografía muestreados. Una caída de la pérdida no demuestra,
por sí sola, mejor decoración ni conservación espacial. Se conserva la serie completa y una media móvil descriptiva.

In [ ]:
loss = training_report['losses']
steps = np.array([r['step'] for r in loss])
values = np.array([r['loss'] for r in loss])
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(steps, values, alpha=.3, label='Pérdida por iteración')
if len(values) >= 20:
    ax.plot(steps[19:], np.convolve(values, np.ones(20)/20, mode='valid'), label='Media móvil de 20')
ax.set(xlabel='Iteración', ylabel='MSE de predicción del ruido', title='Entrenamiento del adaptador')
ax.legend(); plt.tight_layout(); plt.show()
print('Memoria máxima (GiB):', training_report.get('peak_gpu_gib'))

## 5. Comparación controlada de inferencia

Tres dormitorios de desarrollo × strengths 0.35/0.55/0.75 × escalas LoRA 0.5/0.8/1.0 × semillas 42/123 = 54 resultados.
Se compara con 18 salidas del modelo base. Foto, prompt, pasos, CFG, resolución y scheduler se mantienen controlados.
La semilla se reinicia en cada llamada. Esta muestra es exploratoria; los tres dormitorios se usan para desarrollo.

In [ ]:
if IN_KAGGLE:
    evaluator = next(SEARCH.rglob('evaluate_lora.py'))
    runpy.run_path(str(evaluator), run_name='__main__')
    result_path = ROOT / 'lora_evaluation_report.json'
else:
    result_path = ROOT / 'trained_evaluation' / 'lora_evaluation_report.json'
evaluation = json.loads(result_path.read_text())
assert evaluation['status'] == 'complete', evaluation.get('error')
assert len(evaluation['results']) == 54
print('Generaciones con LoRA:', len(evaluation['results']))

## 6. Magnitud del cambio

MAE RGB y SSIM se calculan a 256×256 para pares comparables. Describen cambio respecto de la entrada;
no acreditan realismo, buen diseño ni preservación correcta de puertas y ventanas. Se acompañan de revisión visual.

In [ ]:
summary = []
for strength in [.35, .55, .75]:
    for scale in [.5, .8, 1.0]:
        selected = [r for r in evaluation['results'] if r['strength'] == strength and r['lora_scale'] == scale]
        summary.append({'strength': strength, 'lora_scale': scale, 'n': len(selected),
            'ssim_mean': round(float(np.mean([r['ssim_to_input'] for r in selected])), 3),
            'rgb_mae_mean': round(float(np.mean([r['rgb_mae_0_1'] for r in selected])), 3),
            'seconds_mean': round(float(np.mean([r['seconds'] for r in selected])), 2)})
for row in summary: print(row)

## 7. Revisión humana

En las rejillas, cada fila corresponde a un strength y cada columna a una escala del adaptador.
Valorar por separado: apariencia Luxury, calidad visual y conservación de cama, ventanas, puertas y perspectiva.
La selección final requiere esta revisión, no solamente ordenar una métrica.

In [ ]:
folder = ROOT if IN_KAGGLE else ROOT / 'trained_evaluation'
for grid in sorted(folder.glob('*_parameter_grid.jpg')):
    display(DisplayImage(filename=str(grid)))

## 8. Evidencia y límites

- Los modelos base, revisiones, prompts, semillas, configuraciones, tiempos y memoria están en JSON/CSV.
- Los 10 ejemplos de prueba del estilo se reservan para el cierre; no se presentan como evaluación realizada aquí.
- Predominio de fotografías hoteleras, etiquetas Luxury subjetivas y agrupación aproximada de habitaciones.
- No hay garantía de conservar geometría. Una propuesta es una referencia visual, no un plano constructivo.
- Las fotografías conservan sus licencias individuales; créditos en CREDITS.md.
- Qwen se evaluó en otro experimento: descripción/JSON y resumen de decisiones son tareas distintas.

## 9. Evaluación humana registrada

Fabryzzio prefiere la opción 5 (SDXL base, strength 0.75) y después la 6 (LoRA propio, strength 0.75, escala 0.8) de la lámina comparacion_C61546110.jpg. Las opciones 1–4 no cumplieron sus expectativas. Este voto se limita a esa imagen y semilla; no acredita conservación geométrica. El registro íntegro está en REVISION_HUMANA.md y revision_humana.json. La siguiente comparación mantiene strength 0.75 en los dos modelos.